# Registration and metrics with module

In [ ]:
import os 
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from skimage.transform import resize, AffineTransform, warp

from tifffile import imread

from __future__ import print_function

In [ ]:
import sys
sys.path.insert(0, r"D:\Masters_Medical_Informatics_Larger_Files\Thesis\Working_Folder\Masters-Thesis\src\registration")

In [ ]:
import importlib, registration
importlib.reload(registration)
importlib.reload(registration.reg)
importlib.reload(registration.metrics)
importlib.reload(registration.regPipeline)

In [ ]:
def load_image_data(folder_path):

    img_data = []
    label_data = []
    
    for _, _, files in os.walk(folder_path):
        for index, file in enumerate(files):
            if file.endswith(".tif"): 
                image_path = os.path.join(folder_path, file)
                img_raw = imread(image_path)
                img = np.array(img_raw)
                img_data.append(img)

    return img_data

def display_image(img):
    img = img.astype(float)
    img = img / img.max()
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.colorbar()                 
    plt.show()

## H&E as moving image

#### Load data

In [ ]:
hne_px_sz = 0.5023
dapi_px_sz = 0.209877

scale = hne_px_sz / dapi_px_sz

# load dapi
dapi_img_data = load_image_data("../../../../Thesis_Data/for_valis/CRC027/cropped/dapi")
dapi1_init = resize(dapi_img_data[0], (int(dapi_img_data[0].shape[0]/scale), int(dapi_img_data[0].shape[1]/scale)), anti_aliasing=True)
dapi2_init = resize(dapi_img_data[1], (int(dapi_img_data[1].shape[0]/scale), int(dapi_img_data[1].shape[1]/scale)), anti_aliasing=True)
dapi1_init, dapi2_init = dapi1_init*255, dapi2_init*255

# load hne
hne_image_data = load_image_data("../../../../Thesis_Data/for_valis/CRC027/cropped/hne")
hne1_init = hne_image_data[0]
hne2_init = hne_image_data[1]

display_image(dapi1_init)
display_image(hne1_init)

#### Preprocess images

In [ ]:
hne1_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne1_init)
hne2_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne2_init)
display_image(hne1_deconv)
display_image(hne2_deconv)

#### Register images

In [ ]:
def overlay_images(dapi_img, hne_img, title='overlay'):
    h, w = dapi_img.shape
    plt.imshow(dapi_img, cmap='Greens', alpha=1)
    plt.imshow(hne_img, cmap='Reds', alpha=0.4)
    plt.title(title)
    plt.axis([0, w, h, 0])
    
    legend_patches = [
        Patch(color='green', label='DAPI', alpha=0.8),
        Patch(color='red', label='HnE', alpha=0.4)
    ]
    plt.legend(handles=legend_patches)

def display_img_comparison(dapi_img, hne_img, hne_deconv, registered_hne_imgs):
    no_comparisons = len(registered_hne_imgs)
    if 'intensity based' in registered_hne_imgs:
        no_comparisons += len(registered_hne_imgs['intensity based']) - 1

    h, w = dapi_img.shape
    
    plt.figure(figsize=((no_comparisons+3)*3, 6))

    plt.subplot(1, no_comparisons + 3, 1)
    plt.title('DAPI image')
    plt.imshow(dapi_img, cmap='gray')
    plt.axis([0, w, h, 0])

    plt.subplot(1, no_comparisons + 3, 2)
    plt.title('HnE image')
    plt.imshow(hne_img, cmap='gray')
    plt.axis([0, w, h, 0])

    plt.subplot(1, no_comparisons + 3, 3)
    plt.title('Initial Overlay')
    overlay_images(dapi_img, hne_deconv, title='initial overlay')

    for key, value in registered_hne_imgs.items():
        if key == 'intensity based':
            for key1, value1 in value.items():
                plt.subplot(1, no_comparisons + 3, list(value.keys()).index(key1) + 5)
                overlay_images(dapi_img, value1, title=key1)
        else:
            plt.subplot(1, no_comparisons + 3, list(registered_hne_imgs.keys()).index(key) + 4)
            overlay_images(dapi_img, value, title=key)

    plt.tight_layout()
    plt.show()

In [ ]:
# initial feature based similarity transform & advanced feature based projective transform
img1_tforms, img1_registered, img1_tre_pts = registration.register_DAPI_HnE(dapi1_init, hne1_deconv, adv_tform='feature', feature_tform='projective')
display_img_comparison(dapi1_init, hne1_init, hne1_deconv,img1_registered)
tre = registration.compute_TRE(img1_tforms, img1_tre_pts, dapi1_init)
mi = registration.compute_mutual_information(dapi1_init, hne1_deconv, img1_registered)

In [ ]:
# initial feature based similarity transform & advanced feature based affine transform
img1_tforms, img1_registered, img1_tre_pts = registration.register_DAPI_HnE(dapi1_init, hne1_deconv, adv_tform='feature', feature_tform='affine')
display_img_comparison(dapi1_init, hne1_init, hne1_deconv,img1_registered)
tre = registration.compute_TRE(img1_tforms, img1_tre_pts, dapi1_init)
mi = registration.compute_mutual_information(dapi1_init, hne1_deconv, img1_registered)

In [ ]:
# initial feature based similarity transform & follow-up intensity based rigid-affine-bspline transform
img1_tforms, img1_registered, img1_tre_pts = registration.register_DAPI_HnE(dapi1_init, hne1_deconv, adv_tform='intensity', intensity_tform='r-af-bs', mpp=0.5023)
display_img_comparison(dapi1_init, hne1_init, hne1_deconv,img1_registered)
tre = registration.compute_TRE(img1_tforms, img1_tre_pts, dapi1_init, mpp=0.5023)
mi = registration.compute_mutual_information(dapi1_init, hne1_deconv, img1_registered)

In [ ]:
# initial feature based similarity transform 
img1_tforms, img1_registered, img1_tre_pts = registration.register_DAPI_HnE(dapi1_init, hne1_deconv)
display_img_comparison(dapi1_init, hne1_init, hne1_deconv,img1_registered)
tre = registration.compute_TRE(img1_tforms, img1_tre_pts, dapi1_init)
mi = registration.compute_mutual_information(dapi1_init, hne1_deconv, img1_registered)

In [ ]:
# initial feature based similarity transform & advanced feature based affine transform
img2_tforms, img2_registered, img2_tre_pts = registration.register_DAPI_HnE(dapi2_init, hne2_deconv, adv_tform='feature', feature_tform='affine')
display_img_comparison(dapi2_init, hne2_init, hne2_deconv,img2_registered)
tre = registration.compute_TRE(img2_tforms, img2_tre_pts, dapi2_init)
mi = registration.compute_mutual_information(dapi2_init, hne2_deconv, img2_registered)

In [ ]:
# initial feature based similarity transform & follow-up intensity based bspline transform
img2_tforms, img2_registered, img2_tre_pts = registration.register_DAPI_HnE(dapi2_init, hne2_deconv, adv_tform='intensity', intensity_tform='bspline', mpp=0.5023)
display_img_comparison(dapi2_init, hne2_init, hne2_deconv,img2_registered)
tre = registration.compute_TRE(img2_tforms, img2_tre_pts, dapi2_init, mpp=0.5023)
mi = registration.compute_mutual_information(dapi2_init, hne2_deconv, img2_registered)

In [ ]:
# initial feature based similarity transform
hne1_applying_tform = AffineTransform(scale=(1.0, 1.0), rotation=0.5, translation=(100, -200))
hne1_testing_img = warp(hne1_deconv, hne1_applying_tform)
img1_test_tforms, img1_test_registered, img1_test_tre_pts = registration.register_DAPI_HnE(dapi1_init, hne1_testing_img, adv_tform='feature', feature_tform='affine')
display_img_comparison(dapi1_init, hne1_testing_img, hne1_testing_img,img1_test_registered)
mi = registration.compute_mutual_information(dapi1_init, hne1_testing_img, img1_test_registered)

## DAPI as moving image

#### Load data

In [ ]:
# load dapi
dapim_img_data = load_image_data("../../../../Thesis_Data/for_valis/CRC027/cropped/dapi")
dapi1m_init = dapim_img_data[0]
dapi2m_init = dapim_img_data[1]

# load hne
hnef_image_data = load_image_data("../../../../Thesis_Data/for_valis/CRC027/cropped/hne")
hne1f_init = resize(hnef_image_data[0], (int(hnef_image_data[0].shape[0]*scale), int(hnef_image_data[0].shape[1]*scale)), anti_aliasing=True)
hne2f_init = resize(hnef_image_data[1], (int(hnef_image_data[1].shape[0]*scale), int(hnef_image_data[1].shape[1]*scale)), anti_aliasing=True)
hne1f_init, hne2f_init = hne1f_init*255, hne2f_init*255

display_image(dapi1m_init)
display_image(hne1f_init)

#### Preprocess images

In [ ]:
hne1f_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne1f_init)
hne2f_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne2f_init)
display_image(hne1f_deconv)
display_image(hne2f_deconv)

#### Register images

In [ ]:
def overlay_images_hnef(hne_img, dapi_img, hne_deconv, title='overlay'):
    h, w = hne_deconv.shape
    plt.imshow(dapi_img, cmap='Greens', alpha=1)
    plt.imshow(hne_img, cmap='Reds', alpha=0.4)
    plt.title(title)
    plt.axis([0, w, h, 0])
    
    legend_patches = [
        Patch(color='green', label='DAPI', alpha=0.8),
        Patch(color='red', label='HnE', alpha=0.4)
    ]
    plt.legend(handles=legend_patches)

def display_img_comparison_hnef(dapi_img, hne_img, hne_deconv, registered_dapi_imgs):
    no_comparisons = len(registered_dapi_imgs)
    if 'intensity based' in registered_dapi_imgs:
        no_comparisons += len(registered_dapi_imgs['intensity based']) - 1

    h, w = hne_deconv.shape
    
    plt.figure(figsize=((no_comparisons+3)*3, 6))

    plt.subplot(1, no_comparisons + 3, 1)
    img = hne_img.astype(float)
    img = img / img.max()
    plt.title('HnE image')
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.axis([0, w, h, 0])

    plt.subplot(1, no_comparisons + 3, 2)
    plt.title('DAPI image')
    plt.imshow(dapi_img, cmap='gray')
    plt.axis([0, w, h, 0])

    plt.subplot(1, no_comparisons + 3, 3)
    plt.title('Initial Overlay')
    overlay_images_hnef(hne_deconv, dapi_img, hne_deconv, title='initial overlay')

    for key, value in registered_dapi_imgs.items():
        if key == 'intensity based':
            for key1, value1 in value.items():
                plt.subplot(1, no_comparisons + 3, list(value.keys()).index(key1) + 5)
                overlay_images_hnef(hne_deconv, value1, hne_deconv, title=key1)
        else:
            plt.subplot(1, no_comparisons + 3, list(registered_dapi_imgs.keys()).index(key) + 4)
            overlay_images_hnef(hne_deconv, value, hne_deconv, title=key)

    plt.tight_layout()
    plt.show()

In [ ]:
# initial feature based similarity transform & advanced feature based affine transform
img1_tforms, img1_registered, img1_tre_pts = registration.register_DAPI_HnE(hne1f_deconv, dapi1m_init, adv_tform='feature', feature_tform='affine')
display_img_comparison_hnef(dapi1m_init, hne1f_init, hne1f_deconv, img1_registered)
mi = registration.compute_mutual_information(hne1f_deconv, dapi1m_init, img1_registered)

In [ ]:
# initial feature based similarity transform & advanced feature based affine transform
img1_tforms, img1_registered, img1_tre_pts = registration.register_DAPI_HnE(hne1f_deconv, dapi1m_init, adv_tform='intensity', intensity_tform='affine', mpp=0.209877)
display_img_comparison_hnef(dapi1m_init, hne1f_init, hne1f_deconv, img1_registered)
mi = registration.compute_mutual_information(hne1f_deconv, dapi1m_init, img1_registered)

In [ ]:
# initial feature based similarity transform & follow-up intensity based bspline transform
#img2_tforms, img2_registered, img2_tre_pts = registration.register_DAPI_HnE(hne2f_deconv, dapi2m_init, adv_tform='intensity', intensity_tform='bspline', mpp=0.209877)
#display_img_comparison_hnef(dapi2m_init, hne2f_init, hne2f_deconv,img2_registered)
#tre = registration.compute_TRE(img2_tforms, img2_tre_pts, hne2f_deconv, mpp=0.209877)

## Registration with regPipeline

In [ ]:
# HnE as moving image
transformation_maps, registered_imgs, tre, mi = registration.registration_pipeline("../../../../Thesis_Data/for_valis/CRC027/cropped/dapi/dapi_cropped_1.tif", 
                                                                                   "../../../../Thesis_Data/for_valis/CRC027/cropped/hne/hne_cropped_1.tif", 0.209877, 0.5023, fixed_img='multiplexed')

In [ ]:
# DAPI as moving image
transformation_maps, registered_imgs, tre, mi = registration.registration_pipeline("../../../../Thesis_Data/for_valis/CRC027/cropped/hne/hne_cropped_1.tif", 
                                                                                   "../../../../Thesis_Data/for_valis/CRC027/cropped/dapi/dapi_cropped_1.tif", 0.5023, 0.209877, fixed_img='hne')